# 🔬 UCF-101 Human Action Recognition — Compact 3D CNN
**Train · Validate · Test · Analyse · Grad-CAM** | T4 Mixed Precision | Checkpoint Resume

> Runtime → Change runtime type → **GPU (T4)**

All outputs saved to **Google Drive** at `My Drive/UCF101/`

In [ ]:
import subprocess, sys, os
try: subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError: print("ℹ️ nvidia-smi not found — CPU runtime")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "opencv-python-headless", "scikit-learn", "seaborn", "tqdm",
    "thop", "imageio[ffmpeg]", "tensorboard", "decord"], check=True)
subprocess.run(["apt-get", "install", "-qq", "graphviz"], check=True)

import torch
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
print(f"💻 Device: {DEVICE.upper()}", end="")
if HAS_GPU:
    print(f" | GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.set_float32_matmul_precision('high')
    print("⚡ CUDNN benchmark ON | TF32 matmul ON")
else:
    print(" | ⚠️ No GPU — training will be SLOW but functional")

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/UCF101")
CACHE_DIR   = BASE / "cache"
DATASET_DIR = BASE / "dataset"
CKPT_DIR    = BASE / "checkpoints";    CKPT_DIR.mkdir(parents=True, exist_ok=True)
SAVED_DIR   = BASE / "saved_model";    SAVED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR     = BASE / "figures";        FIG_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR = BASE / "metrics";        METRICS_DIR.mkdir(parents=True, exist_ok=True)
TB_DIR      = BASE / "tensorboard";    TB_DIR.mkdir(parents=True, exist_ok=True)
GRADCAM_DIR = BASE / "gradcam";        GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Base: {BASE}")
print(f"  Cache:   {CACHE_DIR} (exists={CACHE_DIR.exists()})")
print(f"  Dataset: {DATASET_DIR} (exists={DATASET_DIR.exists()})")

In [ ]:
EXPERIMENT  = "ucf101_3dcnn"
EPOCHS      = 30          # With BatchNorm3D, the model converges in 30 epochs instead of 100!
BATCH_SIZE  = 8           # Reduced physical batch size to easily fit within 6GB VRAM GTX 1660
ACCUMULATION_STEPS = 4    # 8 * 4 = 32 virtual batch size for identical gradient updates
N_FRAMES    = 10          # Paper §III-B
FRAME_STEP  = 15          # Δt temporal stride
IMG_SIZE    = (224, 224)
LR          = 5e-4        # Higher learning rate is now fully stable with BatchNorm3d (5x speedup!)
ADAM_BETAS  = (0.9, 0.999)
VAL_SPLIT   = 0.20        # 80/20 stratified (Paper §III-B.4)
SEED        = 42
DROPOUT     = 0.3         # Paper §III-C
USE_AMP     = HAS_GPU     # AMP only works on CUDA
NUM_WORKERS = 0           # Set to 0 to bypass Colab's extremely tight /dev/shm shared memory limit (prevents worker SIGKILL crashes)
PREFETCH    = 2
CONV_FILTERS = [32, 64, 128]  # Paper §III-C
USE_CACHE    = True           # True = use .npy cache (fast), False = decode raw video
COPY_UNCOMPRESSED_FROM_DRIVE = False  # False = bypass slow shutil.copytree of uncompressed folder (recommended)
CACHE_WORKERS = os.cpu_count() or 2
print(f"⚙️ Batch={BATCH_SIZE} | VirtualBatch={BATCH_SIZE*ACCUMULATION_STEPS} | AMP={USE_AMP} | Cache={'ON' if USE_CACHE else 'OFF'} | CopyUncompressed={COPY_UNCOMPRESSED_FROM_DRIVE} | CacheWorkers={CACHE_WORKERS}")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Compact3DCNN(nn.Module):
    """3-layer 3D CNN — exact reproduction of paper §III-C.
    Conv3D(32)→Pool→Conv3D(64)→Pool→Conv3D(128)→Pool→Dropout→GAP→Dense(N)"""
    def __init__(self, num_classes, dropout=DROPOUT):
        super().__init__()
        # Conv3D -> BatchNorm3D -> ReLU -> MaxPool3D
        self.conv1 = nn.Sequential(
            nn.Conv3d(3, 32, 3, padding=1, bias=False), 
            nn.BatchNorm3d(32),
            nn.ReLU(False), 
            nn.MaxPool3d(2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv3d(32, 64, 3, padding=1, bias=False), 
            nn.BatchNorm3d(64),
            nn.ReLU(False), 
            nn.MaxPool3d(2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv3d(64, 128, 3, padding=1, bias=False), 
            nn.BatchNorm3d(128),
            nn.ReLU(False), 
            nn.MaxPool3d(2)
        )
        self.drop = nn.Dropout(dropout)
        self.gap  = nn.AdaptiveAvgPool3d(1)
        self.fc   = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x); x = self.conv2(x); x = self.conv3(x)
        x = self.drop(x); x = self.gap(x); x = x.view(x.size(0), -1)
        return self.fc(x)

def build_model(n_cls):
    m = Compact3DCNN(n_cls).to(DEVICE)
    p = sum(x.numel() for x in m.parameters())
    print(f"Model: {p:,} params → {DEVICE}")
    return m

In [ ]:
import json, random, time, numpy as np, cv2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter
from tqdm.auto import tqdm  # auto-detects Colab → uses widget bars (no line spam)

class CachedNpyDataset(Dataset):
    """Loads pre-extracted .npy frame tensors. Shape: (C,T,H,W) float32.
    Robust against missing or corrupt files on Google Drive."""
    def __init__(self, paths, labels):
        self.paths, self.labels = paths, labels
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            frames = np.load(self.paths[i]).astype(np.float32)
            if frames.ndim == 4 and frames.shape[0] == N_FRAMES:
                frames = torch.from_numpy(frames).permute(3, 0, 1, 2)  # (T,H,W,C)→(C,T,H,W)
            elif frames.ndim == 4 and frames.shape[-1] != 3:
                frames = torch.from_numpy(frames)  # already (C,T,H,W)
            else:
                frames = torch.from_numpy(frames).permute(3, 0, 1, 2)
        except Exception as e:
            # Print warning but do not crash. Return a zero tensor.
            print(f"\n⚠️ Warning: Failed to load {self.paths[i]} ({e}). Returning zero tensor.")
            frames = torch.zeros((3, N_FRAMES, 224, 224), dtype=torch.float32)
        return frames, self.labels[i]

EXTS = {".avi", ".mp4", ".mov", ".mkv", ".webm"}

HAS_DECORD = False
try:
    import decord
    HAS_DECORD = True
except Exception:
    pass

class VideoDataset(Dataset):
    """Fallback: decode video on-the-fly if no cache."""
    def __init__(self, paths, labels, is_train=True):
        self.paths, self.labels, self.train = paths, labels, is_train
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        frames = self._load(self.paths[i])
        return torch.from_numpy(frames).permute(3, 0, 1, 2).float(), self.labels[i]
    def _load(self, path):
        if HAS_DECORD:
            try:
                return self._load_decord(path)
            except Exception:
                return self._load_cv2(path)
        else:
            return self._load_cv2(path)
    def _load_decord(self, path):
        import decord
        vr = decord.VideoReader(str(path))
        total = len(vr)
        need = 1 + (N_FRAMES - 1) * FRAME_STEP
        start = random.randint(0, max(0, total - need)) if self.train and total > need else 0
        indices = [min(total - 1, start + k * FRAME_STEP) for k in range(N_FRAMES)]
        frames = vr.get_batch(indices).asnumpy()
        out = []
        for f in frames:
            f_float = f.astype(np.float32) / 255.0
            h, w = f_float.shape[:2]; s = min(224/h, 224/w); nh, nw = int(h*s), int(w*s)
            f_resized = cv2.resize(f_float, (nw, nh), cv2.INTER_LINEAR)
            ph, pw = 224-nh, 224-nw
            f_padded = cv2.copyMakeBorder(f_resized, ph//2, ph-ph//2, pw//2, pw-pw//2, cv2.BORDER_CONSTANT)
            out.append(f_padded)
        return np.array(out, np.float32)
    def _load_cv2(self, path):
        cap = cv2.VideoCapture(str(path))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        need = 1 + (N_FRAMES - 1) * FRAME_STEP
        start = random.randint(0, max(0, total - need)) if self.train and total > need else 0
        
        frames = []
        if start > 0:
            cap.set(cv2.CAP_PROP_POS_FRAMES, start)
            
        target_indices = {k * FRAME_STEP for k in range(N_FRAMES)}
        max_idx = (N_FRAMES - 1) * FRAME_STEP
        
        curr_idx = 0
        while len(frames) < N_FRAMES:
            ok, f = cap.read()
            if not ok:
                break
            if curr_idx in target_indices:
                f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
                h, w = f.shape[:2]; s = min(224/h, 224/w); nh, nw = int(h*s), int(w*s)
                f = cv2.resize(f, (nw, nh), cv2.INTER_LINEAR)
                ph, pw = 224-nh, 224-nw
                f = cv2.copyMakeBorder(f, ph//2, ph-ph//2, pw//2, pw-pw//2, cv2.BORDER_CONSTANT)
                frames.append(f)
            curr_idx += 1
            if curr_idx > max_idx:
                break
                
        while len(frames) < N_FRAMES:
            frames.append(np.zeros((224, 224, 3), np.float32))
            
        cap.release()
        return np.array(frames, np.float32)

def build_loaders():
    """Use cached .npy or raw video based on USE_CACHE toggle."""
    use_cache = USE_CACHE and Path("/content/cache").exists() and any(Path("/content/cache").iterdir())
    data_root = Path("/content/cache") if use_cache else DATASET_DIR
    
    ext_filter = {".npy"} if use_cache else EXTS
    print(f"{'⚡ Using cached .npy' if use_cache else '📹 Decoding raw video'} from {data_root}")

    classes = sorted([d.name for d in data_root.iterdir() if d.is_dir() and not d.name.startswith(".")])
    c2i = {c: i for i, c in enumerate(classes)}
    paths, labels = [], []
    for c, i in tqdm(c2i.items(), desc="Indexing"):
        for f in (data_root / c).iterdir():
            if f.suffix.lower() in ext_filter:
                paths.append(str(f)); labels.append(i)

    print(f"Dataset: {len(paths)} samples | {len(classes)} classes")

    # Filter sparse classes
    cnt = Counter(labels)
    min_s = max(2, int(1 / VAL_SPLIT) + 1)
    ok = {l for l, n in cnt.items() if n >= min_s}
    if len(ok) < len(set(labels)):
        filt = [(p, l) for p, l in zip(paths, labels) if l in ok]
        paths = [p for p, _ in filt]; labels = [l for _, l in filt]
        o2n = {o: n for n, o in enumerate(sorted(ok))}
        labels = [o2n[l] for l in labels]
        classes = [classes[o] for o in sorted(ok)]

    tr_p, vl_p, tr_l, vl_l = train_test_split(
        paths, labels, test_size=VAL_SPLIT, random_state=SEED, stratify=labels)
    print(f"Split: {len(tr_p)} train / {len(vl_p)} val")

    DS = CachedNpyDataset if use_cache else VideoDataset
    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=HAS_GPU,
              prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None,
              persistent_workers=True if NUM_WORKERS > 0 else False)
    if use_cache:
        tr = DataLoader(DS(tr_p, tr_l), shuffle=True, drop_last=True, **kw)
        vl = DataLoader(DS(vl_p, vl_l), shuffle=False, **kw)
    else:
        tr = DataLoader(DS(tr_p, tr_l, is_train=True), shuffle=True, drop_last=True, **kw)
        vl = DataLoader(DS(vl_p, vl_l, is_train=False), shuffle=False, **kw)
    return tr, vl, classes

In [ ]:
# Bypasses slow I/O by pre-extracting/pre-copying via a consolidated tarball from Google Drive.
# If no cache exists, it downloads the raw dataset, extracts it, pre-processes it to uint8 format,
DRIVE_TAR = BASE / "cache_uint8.tar.gz"          # ← compressed now!
LOCAL_CACHE = Path("/content/cache")
LOCAL_DATA = Path("/content/UCF-101")
RAR_PATH = Path("/content/UCF101.rar")

def _disk_free_gb(path="/content"):
    """Return free disk space in GB for the filesystem containing *path*."""
    import shutil
    return shutil.disk_usage(path).free / (1024**3)

def _dir_size_gb(path):
    """Return total size of a directory tree in GB."""
    total = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())
    return total / (1024**3)

if USE_CACHE:
    # A local cache is only valid if it exists and contains processed .npy files
    local_cache_valid = LOCAL_CACHE.exists() and any(LOCAL_CACHE.glob("**/*.npy"))
    
    if not local_cache_valid:
        if LOCAL_CACHE.exists():
            print(f"🧹 Clearing incomplete/empty local cache directory: {LOCAL_CACHE}")
            import shutil
            shutil.rmtree(LOCAL_CACHE, ignore_errors=True)
            
        # ══════════════════════════════════════════════════════════════
        # PRIORITY 1: .7z archive (locally converted, smallest file)
        # ══════════════════════════════════════════════════════════════
        DRIVE_7Z_OPTIONS = [
            BASE / "archived" / "UCF101_uint8.7z",                        # user's primary location
            BASE / "UCF101_uint8.7z",
            Path("/content/drive/MyDrive/UCF101/archived/UCF101_uint8.7z"),
            Path("/content/drive/MyDrive/UCF101/UCF101_uint8.7z"),
            Path("/content/drive/MyDrive/UCF101_uint8.7z"),
        ]
        found_7z = None
        for opt in DRIVE_7Z_OPTIONS:
            try:
                if opt.exists():
                    found_7z = opt
                    break
            except Exception:
                pass
        
        # Broad search fallback for .7z
        if not found_7z:
            import glob
            for pattern in ["/content/drive/MyDrive/UCF101/**/*uint8*.7z",
                            "/content/drive/MyDrive/*uint8*.7z"]:
                matches = glob.glob(pattern, recursive=True)
                if matches:
                    found_7z = Path(matches[0])
                    break

        if found_7z:
            sz = found_7z.stat().st_size / (1024**3)
            print(f"📦 Found .7z cache on Drive: {found_7z}  ({sz:.2f} GB)")
            
            # Install 7z if not available
            import subprocess
            if subprocess.run(["which", "7z"], capture_output=True).returncode != 0:
                print("Installing p7zip…")
                subprocess.run(["apt-get", "install", "-qq", "p7zip-full"], check=True)
            
            # Copy .7z to local SSD first (Drive FUSE is slow for random reads)
            local_7z = Path("/content/UCF101_uint8.7z")
            print(f"Copying .7z to local SSD ({sz:.1f} GB)…")
            import shutil
            shutil.copy(str(found_7z), str(local_7z))
            
            # Extract to /content/cache
            LOCAL_CACHE.mkdir(exist_ok=True, parents=True)
            print("Extracting .7z to /content/cache…")
            result = subprocess.run(
                ["7z", "x", str(local_7z), f"-o{LOCAL_CACHE}", "-y"],
                capture_output=True, text=True
            )
            if result.returncode == 0:
                print("✓ .7z cache extracted to /content/cache!")
                local_cache_valid = True
            else:
                print(f"⚠️ 7z extraction failed: {result.stderr[:200]}")
            
            # Delete local .7z copy to free disk
            local_7z.unlink(missing_ok=True)
            print(f"✓ Cleaned up local .7z copy. Free disk: {_disk_free_gb():.1f} GB")

        # ══════════════════════════════════════════════════════════════
        # PRIORITY 2: .tar.gz / .tar archives
        # ══════════════════════════════════════════════════════════════
        if not local_cache_valid:
            import glob
            DRIVE_TAR_OPTIONS = [
                DRIVE_TAR,                                                    # .tar.gz (new)
                BASE / "cache_uint8.tar",                                     # .tar (legacy)
                Path("/content/drive/MyDrive/cache_uint8.tar.gz"),
                Path("/content/drive/MyDrive/cache_uint8.tar"),
                Path("/content/drive/MyDrive/UCF101/cache_uint8.tar.gz"),
                Path("/content/drive/MyDrive/UCF101/cache_uint8.tar"),
                Path("/content/drive/MyDrive/UCF101/cache/cache_uint8.tar"),
            ]
            found_tar = None
            for opt in DRIVE_TAR_OPTIONS:
                try:
                    if opt.exists():
                        found_tar = opt
                        break
                except Exception:
                    pass
                    
            # Broad search for .tar / .tar.gz
            if not found_tar:
                for pattern in ["/content/drive/MyDrive/*cache*.tar.gz",
                                "/content/drive/MyDrive/*cache*.tar",
                                "/content/drive/MyDrive/UCF101/*cache*.tar.gz",
                                "/content/drive/MyDrive/UCF101/*cache*.tar"]:
                    matches = glob.glob(pattern)
                    if matches:
                        for match in matches:
                            if "cache" in Path(match).name.lower():
                                found_tar = Path(match)
                                break
                        if found_tar:
                            break
                    
            if found_tar:
                print(f"📦 Found cache tarball on Drive: {found_tar}  ({found_tar.stat().st_size / 1e9:.1f} GB)")
                print("Extracting to local SSD…")
                LOCAL_CACHE.mkdir(exist_ok=True, parents=True)
                import tarfile
                mode = "r:gz" if found_tar.suffix == ".gz" or ".tar.gz" in found_tar.name else "r"
                with tarfile.open(found_tar, mode) as tar:
                    tar.extractall(path="/content/cache")
                print("✓ Cache extracted to /content/cache!")
                local_cache_valid = True
                DRIVE_TAR = found_tar

        # ══════════════════════════════════════════════════════════════
        # PRIORITY 3: Uncompressed cache folder on Drive
        # ══════════════════════════════════════════════════════════════
        if not local_cache_valid and CACHE_DIR.exists() and any(CACHE_DIR.iterdir()) and COPY_UNCOMPRESSED_FROM_DRIVE:
            print(f"Found uncompressed cache on Drive: {CACHE_DIR}")
            print("Copying to local SSD (this can be slow)…")
            import shutil
            shutil.copytree(str(CACHE_DIR), str(LOCAL_CACHE))
            print("✓ Cache copy complete!")
            local_cache_valid = True
            
    # 3. If no preprocessed cache at all, download + extract + cache + pack!
    if not (LOCAL_CACHE.exists() and any(LOCAL_CACHE.glob("**/*.npy"))):
        print("No preprocessed cache found. Downloading and building from scratch…")
        
        # Download UCF-101 raw dataset (~6.5 GB) if not present locally
        if not LOCAL_DATA.exists():
            print("Downloading UCF-101 (~6.5 GB)…")
            import subprocess
            subprocess.run([
                "wget", "-q", "--show-progress", "--no-check-certificate",
                "-O", str(RAR_PATH),
                "https://www.crcv.ucf.edu/data/UCF101/UCF101.rar"
            ], check=True)
            print("Extracting raw videos…")
            from tqdm import tqdm
            
            process = subprocess.Popen(
                ["unrar", "x", "-o+", str(RAR_PATH), "/content/"],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True
            )
            with tqdm(total=13320, desc="Extracting", unit="file") as pbar:
                for line in process.stdout:
                    if line.startswith("Extracting  "):
                        pbar.update(1)
            process.wait()
            RAR_PATH.unlink(missing_ok=True)
            print("✓ Extraction complete! Raw videos ready at /content/UCF-101.")
        
        # Pre-process raw videos to uint8 cache locally
        print("Generating local uint8 frame cache…")
        LOCAL_CACHE.mkdir(exist_ok=True)
        classes_to_cache = sorted([d.name for d in LOCAL_DATA.iterdir() if d.is_dir()])
        
        def extract_and_save(video_path, cls_name):
            out_dir = LOCAL_CACHE / cls_name
            out_dir.mkdir(exist_ok=True, parents=True)
            out_path = out_dir / f"{Path(video_path).stem}.npy"
            if out_path.exists(): return True
            try:
                ds = VideoDataset([str(video_path)], [0], is_train=False)
                # Force cv2 for 100% stable, deadlock-free parallel processing in multiprocessing pools
                frames = ds._load_cv2(video_path)
                frames_uint8 = np.clip(frames * 255.0, 0, 255).astype(np.uint8)
                np.save(out_path, frames_uint8)
                return True
            except Exception as e:
                return False

        all_vids = []
        for c in classes_to_cache:
            for f in (LOCAL_DATA / c).iterdir():
                if f.suffix.lower() in EXTS: all_vids.append((f, c))
                
        print(f"Caching {len(all_vids)} videos locally to {LOCAL_CACHE} using {CACHE_WORKERS} CPU workers...")
        from multiprocessing import Pool
        with Pool(CACHE_WORKERS) as p:
            r = list(tqdm(p.starmap(extract_and_save, all_vids), total=len(all_vids), desc="Generating .npy Cache"))
        print(f"✓ Cached {sum(r)}/{len(all_vids)} videos.")
        
        # ── Delete raw videos BEFORE creating tar to free disk ──
        print("🧹 Cleaning up raw videos to free disk for compression…")
        import shutil
        RAR_PATH.unlink(missing_ok=True)
        if LOCAL_DATA.exists():
            shutil.rmtree(LOCAL_DATA)
        print(f"✓ Freed disk. Available: {_disk_free_gb():.1f} GB")
        
        # ── Create COMPRESSED tar.gz locally ──
        cache_size = _dir_size_gb(LOCAL_CACHE)
        free_disk = _disk_free_gb()
        # Compressed tar.gz is typically 40-60% of uncompressed uint8 npy data
        estimated_tar = cache_size * 0.55
        
        print(f"\n📦 Compressing cache to tar.gz on local SSD…")
        print(f"   Cache size:    {cache_size:.1f} GB")
        print(f"   Est. tar.gz:   ~{estimated_tar:.1f} GB")
        print(f"   Free disk:     {free_disk:.1f} GB")
        
        if free_disk < estimated_tar + 1.0:
            print(f"⚠️  Not enough local disk to create tar.gz ({free_disk:.1f} GB free, need ~{estimated_tar + 1:.1f} GB)")
            print("   Skipping Drive persistence. Cache is still valid for THIS session.")
            print("   Next runtime will re-build the cache (faster with the download already done).")
        else:
            local_tar = Path("/content/cache_uint8.tar.gz")
            import tarfile
            with tarfile.open(local_tar, "w:gz", compresslevel=1) as tar:
                # compresslevel=1 → fast compression, still 2-3× size reduction for uint8 npy
                for cls_dir in tqdm(sorted(LOCAL_CACHE.iterdir()), desc="Compressing", unit="class"):
                    if cls_dir.is_dir():
                        tar.add(str(cls_dir), arcname=cls_dir.name)
            
            tar_size = local_tar.stat().st_size / (1024**3)
            print(f"✓ Compressed: {cache_size:.1f} GB → {tar_size:.1f} GB ({tar_size/cache_size:.0%})")
            
            # ── Copy tar.gz to Google Drive ──
            # Check if Drive has enough space
            try:
                drive_free = _disk_free_gb("/content/drive/MyDrive")
            except Exception:
                drive_free = 999  # can't check, try anyway
                
            if tar_size > drive_free - 1.0:
                print(f"\n⚠️  Google Drive too full! Need {tar_size:.1f} GB but only {drive_free:.1f} GB free.")
                print(f"   The compressed cache is at: {local_tar}")
                print("   Options: 1) Free up Drive space  2) Upgrade storage  3) Download the tar.gz manually")
                print("   Training will proceed using the local cache for this session.")
            else:
                print(f"\n📤 Copying {tar_size:.1f} GB tar.gz to Google Drive: {DRIVE_TAR}")
                print("   Please do not close the notebook until this completes.")
                BASE.mkdir(parents=True, exist_ok=True)
                import shutil
                shutil.copy(str(local_tar), str(DRIVE_TAR))
                local_tar.unlink(missing_ok=True)
                print("✓ Tarball copied to Google Drive!")
                
                # Flush Drive
                print("⚡ Flushing Google Drive…")
                try:
                    from google.colab import drive
                    drive.flush_and_unmount()
                    print("✓ Drive flushed and unmounted!")
                    drive.mount("/content/drive")
                    print("✓ Drive re-mounted for training.")
                except Exception as e:
                    print(f"⚠️ Drive flush failed ({e}). Keep Colab open to let background sync finish.")
        

In [ ]:
train_loader, val_loader, class_names = build_loaders()
NUM_CLASSES = len(class_names)
print(f"\n✓ {NUM_CLASSES} classes: {class_names[:5]} …")

In [ ]:
from torch.amp import GradScaler, autocast

BEST_CKPT   = CKPT_DIR / f"{EXPERIMENT}_best.pth"
LATEST_CKPT = CKPT_DIR / f"{EXPERIMENT}_latest.pth"
STATE_JSON  = CKPT_DIR / f"{EXPERIMENT}_state.json"

# ── Scan for existing checkpoints ──
print(f"\n{'─'*50}")
print(f"🔍 Scanning for checkpoints in: {CKPT_DIR}")
for ckpt_file in [LATEST_CKPT, BEST_CKPT]:
    if ckpt_file.exists():
        try:
            ck_info = torch.load(ckpt_file, map_location="cpu", weights_only=False)
            ep = ck_info.get('epoch', '?')
            acc = ck_info.get('best_val_acc', ck_info.get('val_accuracy', 0.0))
            n_cls = ck_info.get('config', {}).get('num_classes', '?')
            sz = ckpt_file.stat().st_size / (1024**2)
            print(f"   ✅ Found: {ckpt_file.name}")
            print(f"      Epoch: {ep} | Best Acc: {acc:.2%} | Classes: {n_cls} | Size: {sz:.1f} MB")
        except Exception as e:
            print(f"   ⚠️  Found: {ckpt_file.name} (corrupt or incompatible: {e})")
    else:
        print(f"   ❌ Not found: {ckpt_file.name}")
print(f"{'─'*50}")

model     = build_model(NUM_CLASSES)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=ADAM_BETAS, eps=1e-8)
scaler    = GradScaler("cuda", enabled=USE_AMP)
criterion = nn.CrossEntropyLoss()

start_epoch  = 1
best_val_acc = 0.0
history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[], "epoch_time":[], "lr":[]}

# Resume: prefer _latest.pth → _best.pth
resume_ckpt = LATEST_CKPT if LATEST_CKPT.exists() else (BEST_CKPT if BEST_CKPT.exists() else None)
if resume_ckpt:
    ck = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    # Verify checkpoint is compatible with current model
    ckpt_classes = ck.get('config', {}).get('num_classes', None)
    if ckpt_classes is not None and ckpt_classes != NUM_CLASSES:
        print(f"⚠️  Checkpoint has {ckpt_classes} classes but current dataset has {NUM_CLASSES}.")
        print(f"   Skipping resume — training from scratch.")
    else:
        try:
            model.load_state_dict(ck["model_state_dict"])
            optimizer.load_state_dict(ck["optimizer_state_dict"])
            best_val_acc = ck.get("best_val_acc", ck.get("val_accuracy", 0.0))
            start_epoch = ck["epoch"] + 1
            print(f"\n🔄 RESUMING from epoch {ck['epoch']} (best_acc={best_val_acc:.2%})")
            print(f"   Checkpoint: {resume_ckpt.name}")
            print(f"   Will continue from epoch {start_epoch} → {EPOCHS}")
        except Exception as e:
            print(f"⚠️  Failed to load checkpoint weights: {e}")
            print(f"   Training from scratch instead.")
else:
    print(f"\n🆕 No checkpoint found — starting fresh training from epoch 1.")

if STATE_JSON.exists():
    history = json.loads(STATE_JSON.read_text())
    keep = start_epoch - 1
    for k in history: history[k] = history[k][:keep]
    print(f"✓ History loaded & truncated to {keep} epochs")

In [ ]:
try:
    import graphviz

    dot = graphviz.Digraph(format='png', graph_attr={
        'rankdir': 'TB', 'bgcolor': 'white', 'dpi': '200',
        'pad': '0.5', 'nodesep': '0.4', 'ranksep': '0.5'
    })
    dot.attr('node', shape='record', style='filled,rounded',
             fontname='Helvetica', fontsize='11', penwidth='1.5')
    dot.attr('edge', color='#555555', penwidth='1.5', arrowsize='0.8')

    # Nodes — paper §III-C architecture
    dot.node('input',  'Input\n(B, 3, 10, 224, 224)',  fillcolor='#dfe6e9', fontcolor='#2d3436')
    dot.node('conv1',  'Conv3D(3→32) + ReLU\nkernel 3×3×3, pad=1\nMaxPool3D(2,2,2)',
             fillcolor='#74b9ff', fontcolor='#2d3436')
    dot.node('s1',     '(B, 32, 5, 112, 112)', fillcolor='#f8f9fa', fontcolor='#636e72', shape='plaintext')
    dot.node('conv2',  'Conv3D(32→64) + ReLU\nkernel 3×3×3, pad=1\nMaxPool3D(2,2,2)',
             fillcolor='#55efc4', fontcolor='#2d3436')
    dot.node('s2',     '(B, 64, 2, 56, 56)', fillcolor='#f8f9fa', fontcolor='#636e72', shape='plaintext')
    dot.node('conv3',  'Conv3D(64→128) + ReLU\nkernel 3×3×3, pad=1\nMaxPool3D(2,2,2)',
             fillcolor='#ffeaa7', fontcolor='#2d3436')
    dot.node('s3',     '(B, 128, 1, 28, 28)', fillcolor='#f8f9fa', fontcolor='#636e72', shape='plaintext')
    dot.node('drop',   f'Dropout({DROPOUT})',  fillcolor='#fab1a0', fontcolor='#2d3436')
    dot.node('gap',    'Global Avg Pool 3D\nAdaptiveAvgPool3d(1)', fillcolor='#a29bfe', fontcolor='white')
    dot.node('s4',     '(B, 128)', fillcolor='#f8f9fa', fontcolor='#636e72', shape='plaintext')
    dot.node('fc',     f'Dense(128 → {NUM_CLASSES})\nLinear + Softmax', fillcolor='#fd79a8', fontcolor='white')
    dot.node('output', f'Output\n(B, {NUM_CLASSES})',  fillcolor='#00b894', fontcolor='white')

    # Edges
    for a, b in [('input','conv1'),('conv1','s1'),('s1','conv2'),('conv2','s2'),
                  ('s2','conv3'),('conv3','s3'),('s3','drop'),('drop','gap'),
                  ('gap','s4'),('s4','fc'),('fc','output')]:
        dot.edge(a, b)

    arch_path = str(FIG_DIR / f"{EXPERIMENT}_architecture")
    dot.render(arch_path, cleanup=True)
    print(f"✓ Architecture diagram → {arch_path}.png")
    from IPython.display import Image, display
    display(Image(filename=arch_path + ".png"))
except Exception as e:
    print(f"Architecture viz skipped: {e}")

# FLOPs count
try:
    from thop import profile
    dummy = torch.randn(1, 3, N_FRAMES, *IMG_SIZE).to(DEVICE)
    flops, params = profile(model, inputs=(dummy,), verbose=False)
    print(f"✓ FLOPs: {flops/1e9:.2f} GFLOPs | Params: {params:,}")
except: pass

In [ ]:
print(f"ℹ️ TensorBoard logs will be saved to: {TB_DIR / EXPERIMENT}")
print("   To view: %load_ext tensorboard")
print(f"   Then:    %tensorboard --logdir {TB_DIR}")

In [ ]:
from torch.utils.tensorboard import SummaryWriter
tb_writer = SummaryWriter(str(TB_DIR / EXPERIMENT))
print(f"✓ TensorBoard writer → {TB_DIR / EXPERIMENT}")

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss = correct = total = 0
    tag = "🏋️ Train" if training else "📊 Val"
    ctx = torch.enable_grad if training else torch.no_grad
    with ctx():
        if training:
            optimizer.zero_grad(set_to_none=True)
            
        for batch_idx, (vids, labels) in enumerate(loader):
            vids, labels = vids.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            
            with autocast("cuda", enabled=USE_AMP):
                out = model(vids)
                loss = criterion(out, labels)
                if training:
                    loss = loss / ACCUMULATION_STEPS
                    
            if training:
                scaler.scale(loss).backward()
                if (batch_idx + 1) % ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(loader):
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
            
            loss_val = (loss.item() * ACCUMULATION_STEPS) if training else loss.item()
            total_loss += loss_val * vids.size(0)
            correct += out.max(1)[1].eq(labels).sum().item()
            total += labels.size(0)
            
            # Print status update every 50 batches using carriage return, completely eliminating line spam!
            if batch_idx % 50 == 0 or (batch_idx + 1) == len(loader):
                print(f"\r    {tag} | Batch {batch_idx}/{len(loader)} | Loss: {total_loss/total:.4f} | Acc: {correct/total:.2%}", end="", flush=True)
                
        # Clean carriage return line when done
        print("\r" + " " * 90 + "\r", end="", flush=True)
            
    return total_loss / total, correct / total

print(f"\n{'='*65}")
print(f"  Training: {EXPERIMENT}")
print(f"  Epochs: {start_epoch} → {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR}")
print(f"  Classes: {NUM_CLASSES} | AMP: {USE_AMP} | Device: {DEVICE}")
print(f"{'='*65}\n")

# Single clean progress bar — NO print spam
epoch_pbar = tqdm(range(start_epoch, EPOCHS + 1), desc="🔥 Training", unit="ep",
                  bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} ep '
                             '[{elapsed}<{remaining}, {rate_fmt}{postfix}]')

for epoch in epoch_pbar:
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, True)
    vl_loss, vl_acc = run_epoch(val_loader, False)
    elapsed = time.time() - t0

    history["train_loss"].append(tr_loss); history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(vl_acc)
    history["epoch_time"].append(elapsed); history["lr"].append(optimizer.param_groups[0]["lr"])

    # TensorBoard
    tb_writer.add_scalars("Loss", {"train": tr_loss, "val": vl_loss}, epoch)
    tb_writer.add_scalars("Accuracy", {"train": tr_acc, "val": vl_acc}, epoch)
    tb_writer.add_scalar("Time/epoch_sec", elapsed, epoch)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                     "optimizer_state_dict": optimizer.state_dict(),
                     "val_accuracy": vl_acc, "best_val_acc": best_val_acc,
                     "class_names": class_names,
                     "config": {"n_frames": N_FRAMES, "frame_step": FRAME_STEP,
                                "img_size": IMG_SIZE, "conv_filters": CONV_FILTERS,
                                "dropout": DROPOUT, "num_classes": NUM_CLASSES}},
                    str(BEST_CKPT))

    # Crash-safe: save every epoch
    torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_accuracy": vl_acc, "best_val_acc": best_val_acc,
                "class_names": class_names}, str(LATEST_CKPT))
    STATE_JSON.write_text(json.dumps(history, indent=2))

    # Update progress bar — this is the ONLY output per epoch (no print!)
    epoch_pbar.set_postfix_str(
        f"val_acc={vl_acc:.2%} | best={best_val_acc:.2%} | loss={vl_loss:.4f} | {elapsed:.0f}s/ep")

tb_writer.close()
print(f"\n✓ Training complete. Best val_acc: {best_val_acc:.2%}")

In [ ]:
import shutil
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                              precision_recall_fscore_support, accuracy_score)

plt.rcParams.update({"font.size": 12, "font.family": "serif", "figure.dpi": 150,
                      "savefig.dpi": 300, "savefig.bbox": "tight",
                      "axes.grid": True, "grid.alpha": 0.3})

ck = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck["model_state_dict"])
model.eval()
print(f"✓ Loaded best checkpoint: epoch {ck['epoch']} | val_acc={ck['val_accuracy']:.2%}")

In [ ]:
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    eval_pbar = tqdm(val_loader, desc="📊 Evaluating", unit="batch",
                     bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')
    for vids, labs in eval_pbar:
        vids = vids.to(DEVICE, non_blocking=True)
        with autocast("cuda", enabled=USE_AMP):
            out = model(vids)
        probs = torch.softmax(out.float(), dim=1)
        preds_batch = probs.argmax(1).cpu().numpy()
        all_preds.extend(preds_batch)
        all_labels.extend(labs.numpy())
        all_probs.extend(probs.cpu().numpy())
        # Live accuracy in progress bar
        running_acc = np.mean(np.array(all_preds) == np.array(all_labels))
        eval_pbar.set_postfix(acc=f"{running_acc:.2%}")

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

In [ ]:
def compute_all_metrics(preds, labels, names):
    p, r, f, s = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    cm = confusion_matrix(labels, preds, labels=range(len(names)))
    spec = []
    for i in range(len(names)):
        tn = cm.sum() - cm[i].sum() - cm[:, i].sum() + cm[i, i]
        fp = cm[:, i].sum() - cm[i, i]
        spec.append(tn / (tn + fp) if (tn + fp) > 0 else 0.)
    acc = accuracy_score(labels, preds)
    mp, mr, mf, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": float(acc),
            "per_class": {names[i]: {"precision": float(p[i]), "recall": float(r[i]),
                "specificity": float(spec[i]), "f1": float(f[i]), "support": int(s[i])}
                for i in range(len(names))},
            "macro": {"precision": float(mp), "recall": float(mr),
                      "specificity": float(np.mean(spec)), "f1": float(mf)},
            "confusion_matrix": cm.tolist()}

metrics = compute_all_metrics(all_preds, all_labels, class_names)
print(f"\n{'='*50}")
print(f"  Overall Accuracy : {metrics['accuracy']:.2%}")
print(f"  Macro F1         : {metrics['macro']['f1']:.4f}")
print(f"  Macro Precision  : {metrics['macro']['precision']:.4f}")
print(f"  Macro Recall     : {metrics['macro']['recall']:.4f}")
print(f"  Macro Specificity: {metrics['macro']['specificity']:.4f}")
print(f"{'='*50}")
print(f"\n{classification_report(all_labels, all_preds, target_names=class_names, zero_division=0)}")

# Save metrics JSON
metrics_path = METRICS_DIR / f"{EXPERIMENT}_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"✓ Metrics → {metrics_path}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history["train_loss"]) + 1)

ax1.plot(ep, history["train_loss"], label="Train Loss", lw=1.5, color="#e17055")
ax1.plot(ep, history["val_loss"], label="Val Loss", lw=1.5, color="#0984e3")
ax1.fill_between(ep, history["train_loss"], alpha=0.1, color="#e17055")
ax1.fill_between(ep, history["val_loss"], alpha=0.1, color="#0984e3")
ax1.set(xlabel="Epoch", ylabel="Loss", title="Training & Validation Loss"); ax1.legend()

ax2.plot(ep, [v*100 for v in history["train_acc"]], label="Train Acc", lw=1.5, color="#e17055")
ax2.plot(ep, [v*100 for v in history["val_acc"]], label="Val Acc", lw=1.5, color="#0984e3")
ax2.fill_between(ep, [v*100 for v in history["train_acc"]], alpha=0.1, color="#e17055")
ax2.fill_between(ep, [v*100 for v in history["val_acc"]], alpha=0.1, color="#0984e3")
ax2.set(xlabel="Epoch", ylabel="Accuracy %", title="Training & Validation Accuracy"); ax2.legend()

plt.tight_layout()
fig.savefig(FIG_DIR / f"{EXPERIMENT}_curves.png"); plt.show(); plt.close()
print(f"✓ Training curves saved")

In [ ]:
cm = np.array(metrics["confusion_matrix"])
n = len(class_names); fs = max(8, n * 0.15); ann = n <= 30; fsz = max(4, 10 - n // 10)

for norm, suffix, title in [(False, "_cm.png", "Confusion Matrix"),
                             (True, "_cm_norm.png", "Confusion Matrix (Normalised)")]:
    d = cm.astype(float) / cm.sum(axis=1, keepdims=True) if norm else cm
    fmt = ".2f" if norm else "d"
    fig, ax = plt.subplots(figsize=(fs, fs))
    sns.heatmap(d, annot=ann, fmt=fmt, cmap="Blues", xticklabels=class_names,
                yticklabels=class_names, ax=ax, annot_kws={"size": fsz}, square=True, linewidths=0.5)
    ax.set(xlabel="Predicted", ylabel="True", title=title)
    plt.xticks(rotation=90, fontsize=fsz); plt.yticks(fontsize=fsz)
    plt.tight_layout(); fig.savefig(FIG_DIR / f"{EXPERIMENT}{suffix}"); plt.show(); plt.close()
print(f"✓ Confusion matrices saved")

In [ ]:
prec = [metrics["per_class"][c]["precision"] for c in class_names]
rec  = [metrics["per_class"][c]["recall"] for c in class_names]
spec = [metrics["per_class"][c]["specificity"] for c in class_names]
f1   = [metrics["per_class"][c]["f1"] for c in class_names]
x = np.arange(n); w = 0.2

fig, ax = plt.subplots(figsize=(max(12, n * 0.3), 6))
for vals, lbl, off, col in [(prec, "Precision", -1.5, "#0984e3"), (rec, "Recall", -0.5, "#00b894"),
                              (spec, "Specificity", 0.5, "#fdcb6e"), (f1, "F1-Score", 1.5, "#e17055")]:
    ax.bar(x + off * w, vals, w, label=lbl, alpha=0.85, color=col)
ax.set(xticks=x, xlabel="Class", ylabel="Score", title="Per-Class Metrics", ylim=(0, 1.1))
ax.set_xticklabels(class_names, rotation=90, fontsize=max(4, 10 - n // 10)); ax.legend()
plt.tight_layout(); fig.savefig(FIG_DIR / f"{EXPERIMENT}_perclass.png"); plt.show(); plt.close()

In [ ]:
f1s = {c: metrics["per_class"][c]["f1"] for c in class_names}
srt = sorted(f1s.items(), key=lambda x: x[1])
bot5 = srt[:5]; top5 = srt[-5:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, ttl, col in [(axes[0], bot5, "Bottom-5 F1", "#e17055"),
                             (axes[1], top5, "Top-5 F1", "#00b894")]:
    cs = [x[0] for x in data]; vs = [x[1] for x in data]
    ax.barh(cs, vs, color=col, alpha=0.85); ax.set(xlim=(0, 1), title=ttl)
plt.tight_layout(); fig.savefig(FIG_DIR / f"{EXPERIMENT}_top_bottom5.png"); plt.show(); plt.close()
print(f"✓ Per-class & Top/Bottom charts saved")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(history["epoch_time"]) + 1), history["epoch_time"], lw=1.5, color="#6c5ce7")
ax.fill_between(range(1, len(history["epoch_time"]) + 1), history["epoch_time"], alpha=0.15, color="#6c5ce7")
avg_t = np.mean(history["epoch_time"])
ax.axhline(avg_t, ls="--", color="#e17055", label=f"Avg: {avg_t:.1f}s")
ax.set(xlabel="Epoch", ylabel="Seconds", title="Epoch Duration"); ax.legend()
plt.tight_layout(); fig.savefig(FIG_DIR / f"{EXPERIMENT}_speed.png"); plt.show(); plt.close()

In [ ]:
import imageio

class GradCAM3D:
    def __init__(self, model):
        self.model = model; self.model.eval()
        self.act = self.grad = None
        model.conv3[0].register_forward_hook(lambda m, i, o: setattr(self, "act", o.detach()))
        model.conv3[0].register_full_backward_hook(lambda m, gi, go: setattr(self, "grad", go[0].detach()))

    def __call__(self, x, cls=None):
        x.requires_grad_(True); self.model.zero_grad()
        with torch.enable_grad():
            out = self.model(x)
            if cls is None: cls = out.argmax(1).item()
            out[0, cls].backward()
        w = self.grad.mean(dim=(2, 3, 4), keepdim=True)
        cam = F.relu((w * self.act).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=(N_FRAMES, *IMG_SIZE), mode="trilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        mn, mx = cam.min(), cam.max()
        return (cam - mn) / (mx - mn + 1e-8), cls

# Generate for first 5 classes
sample_classes = class_names[:5]
gcam = GradCAM3D(model)

for cls in tqdm(sample_classes, desc="Grad-CAM"):
    # Find a sample video from raw dataset or from cache
    if DATASET_DIR.exists() and (DATASET_DIR / cls).exists():
        vids = [f for f in (DATASET_DIR / cls).iterdir() if f.suffix.lower() in EXTS]
        if not vids: continue
        ds = VideoDataset([str(vids[0])], [0], is_train=False)
        frames, _ = ds[0]
    elif CACHE_DIR.exists() and (CACHE_DIR / cls).exists():
        npys = list((CACHE_DIR / cls).glob("*.npy"))
        if not npys: continue
        frames = np.load(npys[0]).astype(np.float32)
        frames = torch.from_numpy(frames).permute(3, 0, 1, 2) if frames.shape[-1] == 3 else torch.from_numpy(frames)
    else: continue

    inp = frames.unsqueeze(0).to(DEVICE)
    hm, pred = gcam(inp.clone())
    fr = frames.permute(1, 2, 3, 0).numpy()
    gif_frames = []
    for t in range(N_FRAMES):
        f = (fr[t] * 255).clip(0, 255).astype(np.uint8)
        h = (hm[t] * 255).astype(np.uint8)
        hc = cv2.cvtColor(cv2.applyColorMap(h, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
        gif_frames.append(cv2.addWeighted(f, 0.6, hc, 0.4, 0))
    out_path = GRADCAM_DIR / f"{cls}_gradcam.gif"
    imageio.mimsave(str(out_path), gif_frames, duration=120, loop=0)
    print(f"  {cls} → predicted: {class_names[pred]}")
    from IPython.display import Image as IPyImage, display
    display(IPyImage(filename=str(out_path)))

print(f"✓ Grad-CAM GIFs → {GRADCAM_DIR}")

In [ ]:
final_save = {
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "num_classes": NUM_CLASSES,
    "best_val_accuracy": best_val_acc,
    "config": {
        "architecture": "Compact3DCNN",
        "conv_filters": CONV_FILTERS,
        "dropout": DROPOUT,
        "n_frames": N_FRAMES,
        "frame_step": FRAME_STEP,
        "img_size": IMG_SIZE,
        "learning_rate": LR,
        "batch_size": BATCH_SIZE,
        "epochs_trained": len(history["train_loss"]),
    },
    "metrics": {
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro"]["f1"],
        "macro_precision": metrics["macro"]["precision"],
        "macro_recall": metrics["macro"]["recall"],
    }
}
save_path = SAVED_DIR / f"{EXPERIMENT}_final.pth"
torch.save(final_save, str(save_path))
print(f"\n✓ Final model saved → {save_path}")
print(f"  Classes: {NUM_CLASSES} | Accuracy: {metrics['accuracy']:.2%} | F1: {metrics['macro']['f1']:.4f}")

In [ ]:
report = {
    "experiment": EXPERIMENT,
    "dataset": "UCF-101",
    "architecture": "Compact3DCNN (Paper §III-C)",
    "total_params": sum(p.numel() for p in model.parameters()),
    "device": DEVICE,
    "hyperparameters": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LR,
        "n_frames": N_FRAMES, "frame_step": FRAME_STEP, "img_size": IMG_SIZE,
        "dropout": DROPOUT, "val_split": VAL_SPLIT, "optimizer": "Adam",
        "mixed_precision": USE_AMP,
    },
    "results": metrics,
    "training_summary": {
        "total_epochs": len(history["train_loss"]),
        "best_val_accuracy": best_val_acc,
        "final_train_loss": history["train_loss"][-1] if history["train_loss"] else None,
        "final_val_loss": history["val_loss"][-1] if history["val_loss"] else None,
        "avg_epoch_time_sec": float(np.mean(history["epoch_time"])) if history["epoch_time"] else None,
    },
    "files_saved": {
        "model": str(save_path),
        "metrics": str(metrics_path),
        "figures": str(FIG_DIR),
        "gradcam": str(GRADCAM_DIR),
        "tensorboard": str(TB_DIR),
    }
}
report_path = METRICS_DIR / f"{EXPERIMENT}_full_report.json"
report_path.write_text(json.dumps(report, indent=2))
print(f"✓ Full research report → {report_path}")

In [ ]:
print(f"\n{'='*65}")
print(f"  🎉 PIPELINE COMPLETE — {EXPERIMENT}")
print(f"{'='*65}")
print(f"  Model    : Compact3DCNN ({sum(p.numel() for p in model.parameters()):,} params)")
print(f"  Accuracy : {metrics['accuracy']:.2%}")
print(f"  Macro F1 : {metrics['macro']['f1']:.4f}")
print(f"  Best Acc : {best_val_acc:.2%}")
print(f"  Saved to : {SAVED_DIR}")
print(f"{'='*65}")
print(f"\n📂 All outputs in Google Drive at: My Drive/UCF101/")
print(f"   ├── checkpoints/   (best + latest .pth)")
print(f"   ├── saved_model/   (final export)")
print(f"   ├── figures/       (curves, CM, per-class, Grad-CAM)")
print(f"   ├── metrics/       (JSON reports)")
print(f"   ├── tensorboard/   (TB logs)")
print(f"   └── gradcam/       (attention GIFs)")